In [1]:
import re
import time
import json
import requests
import pandas as pd
from docx import Document

from rapidfuzz import fuzz
from dateutil import parser as dateparser

import fitz  # PyMuPDF

import importlib
import extractors.acs_pdf as acs_pdf
importlib.reload(acs_pdf)

from extractors.acs_pdf import extract_references_from_acs_pdf
from extractors.acs_title_extractor import ACS_Title_Extractor
from report.acs_report import build_acs_report, PigThresholds
from journal_title.journal_iso4 import best_journal_score, add_iso4_columns
from journal_title.journal_from_raw import extract_journal_from_acs_raw, clean_journal_obs

LOADING ACS REFERENCE EXTRACTOR FROM: C:\Users\robin\Programs\KilbrethsPig\PDF\ACS\extractors\acs_pdf.py
LOADING ACS REFERENCE EXTRACTOR FROM: C:\Users\robin\Programs\KilbrethsPig\PDF\ACS\extractors\acs_pdf.py
LOADING ACS TITLE EXTRACTOR FROM: C:\Users\robin\Programs\KilbrethsPig\PDF\ACS\extractors\acs_title_extractor.py
LOADING KILBRETHS PIG REPORTS FROM: C:\Users\robin\Programs\KilbrethsPig\PDF\ACS\report\acs_report.py
LOADING ISO4 COMPLIANT JOURNAL TITLES FROM: C:\Users\robin\Programs\KilbrethsPig\PDF\ACS\journal_title\journal_iso4.py
LOADING RAW DATA FROM: C:\Users\robin\Programs\KilbrethsPig\PDF\ACS\journal_title\journal_from_raw.py


In [2]:
ARTICLE_TITLE = ACS_Title_Extractor().extract_title("test.pdf")

In [3]:
df_original, report = extract_references_from_acs_pdf("test.pdf", expected_n=55)

In [4]:
DOI_RE = re.compile(r"(10\.\d{4,9}/[^\s\"\'<>]+)", re.IGNORECASE)

def extract_doi_from_raw(raw: str):
    if not isinstance(raw, str):
        return None
    m = DOI_RE.search(raw)
    if not m:
        return None
    return m.group(1).rstrip(").,;]}")


In [5]:
df_original["doi_raw"] = df_original["raw"].apply(extract_doi_from_raw)
df_original["doi_status"] = df_original["doi_raw"].apply(
    lambda d: "OK" if d else "No DOI"
)

df_original["doi_url"] = df_original["doi_raw"].apply(
    lambda d: f"https://doi.org/{d}" if d else None
)

In [6]:
def normalize_doi(doi: str):
    if not isinstance(doi, str) or not doi.strip():
        return None
    doi = doi.strip()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("doi:", "").strip()
    doi = doi.rstrip(").,;]}")
    return doi.lower()

def crossref_find_doi_from_citation(raw: str, timeout=20):
    if not raw:
        return None

    url = "https://api.crossref.org/works"
    params = {"query.bibliographic": raw, "rows": 3}
    headers = {"User-Agent": "kilbreths-pig/0.1 (mailto:YOUR_EMAIL_HERE)"}  # put your email

    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    if r.status_code != 200:
        return None

    items = (r.json().get("message", {}) or {}).get("items", []) or []
    for it in items:
        doi = it.get("DOI")
        if doi:
            return normalize_doi(doi)
    return None

def openalex_find_doi_from_citation(raw: str, timeout=20):
    if not raw:
        return None

    url = "https://api.openalex.org/works"
    params = {"search": raw, "per-page": 3}
    headers = {"User-Agent": "kilbreths-pig/0.1"}

    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    if r.status_code != 200:
        return None

    results = r.json().get("results", []) or []
    for it in results:
        doi = it.get("doi")
        if doi:
            return normalize_doi(doi)
    return None

def fill_missing_dois(df, sleep_s=0.2):
    df = df.copy()

    doi_found = []
    doi_source = []

    for _, row in df.iterrows():
        raw = row.get("raw", "") or ""

        doi = crossref_find_doi_from_citation(raw)
        src = "crossref" if doi else None

        if not doi:
            doi = openalex_find_doi_from_citation(raw)
            src = "openalex" if doi else None

        doi_found.append(doi)
        doi_source.append(src or "none")
        time.sleep(sleep_s)

    df["doi_found"] = doi_found
    df["doi_source"] = doi_source
    df["doi_final"] = df["doi_found"].apply(normalize_doi)
    df["doi_status"] = df["doi_final"].apply(lambda d: "OK" if d else "No DOI")
    df["doi_url_final"] = df["doi_final"].apply(lambda d: f"https://doi.org/{d}" if d else None)

    return df

# Run it:
# df_original = fill_missing_dois(df_original, sleep_s=0.25)
# df_original["doi_status"].value_counts()


In [7]:
df_original = fill_missing_dois(df_original, sleep_s=0.25)

In [8]:
def _safe_first(x):
    return (x[0] if isinstance(x, list) and x else None)

def _acs_author_list(authors):
    """
    ACS-ish author string: 'Last, F. M.; Last, F. M.; ...'
    """
    if not authors:
        return None
    out = []
    for a in authors:
        family = a.get("family")
        given = a.get("given")
        if family and given:
            parts = re.split(r"[\s\-]+", given.strip())
            initials = " ".join([p[0] + "." for p in parts if p])
            out.append(f"{family}, {initials}")
        elif family:
            out.append(f"{family}")
    return "; ".join(out) if out else None

def crossref_fetch_work(doi: str, timeout=20):
    if not doi:
        return None
    url = f"https://api.crossref.org/works/{doi}"
    headers = {"User-Agent": "kilbreths-pig/0.1 (mailto:YOUR_EMAIL_HERE)"}
    r = requests.get(url, headers=headers, timeout=timeout)
    if r.status_code != 200:
        return None
    return (r.json().get("message") or {})

def crossref_to_df_row(ref_id: int, doi: str, msg: dict):
    if not msg:
        return {
            "ref_id": ref_id,
            "doi": doi,
            "ok": False
        }

    issued = (msg.get("issued", {}) or {}).get("date-parts", [[None]])
    year = issued[0][0] if issued and issued[0] else None

    return {
        "ref_id": ref_id,
        "doi": doi,
        "doi_url": f"https://doi.org/{doi}" if doi else None,
        "title": _safe_first(msg.get("title")),
        "authors": _acs_author_list(msg.get("author")),
        "journal": _safe_first(msg.get("container-title")),
        "year": year,
        "volume": msg.get("volume"),
        "issue": msg.get("issue"),
        "pages": msg.get("page"),
        "publisher": msg.get("publisher"),
        "type": msg.get("type"),
        "ok": True
    }

def build_df_doi_from_df_original(df_original: pd.DataFrame, sleep_s=0.15) -> pd.DataFrame:
    rows = []
    for _, r in df_original.iterrows():
        ref_id = int(r["ref_id"])
        doi = r.get("doi_final")
        if not isinstance(doi, str) or not doi:
            rows.append({"ref_id": ref_id, "doi": None, "ok": False})
            continue

        msg = crossref_fetch_work(doi)
        rows.append(crossref_to_df_row(ref_id, doi, msg))
        time.sleep(sleep_s)

    return pd.DataFrame(rows)

In [9]:
df_doi = build_df_doi_from_df_original(df_original, sleep_s=0.2)

In [10]:
df_join = df_original.merge(
    df_doi[["ref_id","title","authors","journal","year","volume","issue","pages","doi"]],
    left_on="ref_id",
    right_on="ref_id",
    how="left",
    suffixes=("_orig","_doi")
)

In [11]:
df_DOI = add_iso4_columns(df_doi) 

In [12]:
def score_row(raw, title, authors, journal, journal_iso4, year):
    s = []

    if title:
        s.append(0.45 * fuzz.token_set_ratio(raw, title))

    if authors:
        s.append(0.25 * fuzz.partial_ratio(raw, authors))

    # --- journal logic ---
    journal_scores = []
    if journal:
        journal_scores.append(fuzz.partial_ratio(raw, journal))
    if journal_iso4:
        journal_scores.append(fuzz.partial_ratio(raw, journal_iso4))

    if journal_scores:
        s.append(0.20 * max(journal_scores))

    if year:
        s.append(0.10 * fuzz.partial_ratio(raw, str(year)))

    return sum(s) / (0.45 + 0.25 + 0.20 + 0.10)


assert df_join["ref_id"].is_unique
assert df_DOI["ref_id"].is_unique

df_join = df_join.merge(
    df_DOI[["ref_id", "journal_iso4"]],
    on="ref_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_doi"),
)

# If df_join already had journal_iso4, prefer the DOI-derived one when present
if "journal_iso4_doi" in df_join.columns:
    df_join["journal_iso4"] = df_join["journal_iso4_doi"].combine_first(df_join.get("journal_iso4"))
    df_join.drop(columns=["journal_iso4_doi"], inplace=True)


df_join["match_score"] = df_join.apply(
    lambda r: score_row(
        r["raw"],
        r["title"],
        r["authors"],
        r["journal"],
        r["journal_iso4"],
        r["year"],
    ),
    axis=1
)

df_join["match_score"].describe()

count     55.000000
mean      86.957720
std        9.739525
min       61.882560
25%       80.209898
50%       87.045455
75%       96.465102
max      100.000000
Name: match_score, dtype: float64

In [13]:
# df_join.sort_values("match_score").head(10)[
#     ["ref_id","match_score","doi","journal","year","title","raw"]
# ]

In [14]:
def _norm_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def _fix_pages(p: str) -> str:
    if not p:
        return ""
    # normalize hyphen variants and remove spaces around dash
    p = p.replace("−", "-").replace("–", "-")
    p = re.sub(r"\s*-\s*", "-", p)
    return p

def format_acs_ref(row) -> str:
    """
    ACS-ish format (not perfect abbreviations):
      (n) Authors. Title. Journal Year, Volume (Issue), Pages. DOI: xxxx.
    """
    ref_id = row.get("ref_id")
    authors = _norm_ws(row.get("authors") or "")
    title = _norm_ws(row.get("title") or "")
    journal = _norm_ws(row.get("journal_iso4") or "")
    year = row.get("year") or ""
    volume = _norm_ws(str(row.get("volume") or ""))
    issue = _norm_ws(str(row.get("issue") or ""))
    pages = _fix_pages(_norm_ws(str(row.get("pages") or "")))
    doi = _norm_ws(row.get("doi") or "")

    bits = []
    if authors:
        bits.append(f"{authors}.")
    if title:
        bits.append(f"{title}.")
    if journal:
        # Journal block: Journal Year, Volume (Issue), Pages.
        j = journal
        y = str(year) if year else ""
        v = volume
        iss = f" ({issue})" if issue and issue.lower() != "none" else ""
        pg = f", {pages}" if pages else ""

        # Build "Journal Year, Volume (Issue), Pages."
        # Handle missing volume gracefully.
        core = j
        if y:
            core += f" {y}"
        if v:
            core += f", {v}{iss}{pg}."
        else:
            core += f"."
        bits.append(core)

    if doi:
        bits.append(f"DOI: {doi}.")

    inner = " ".join(bits).strip()
    return f"({ref_id}) {inner}".strip()

def write_acs_bibliography_docx(df_doi, out_path="acs_bibliography.docx"):
    df = df_doi.sort_values("ref_id").copy()
    doc = Document()
    doc.add_heading("ACS-Style Bibliography (from DOI metadata)", level=1)

    for _, r in df.iterrows():
        doc.add_paragraph(format_acs_ref(r))

    doc.save(out_path)
    return out_path

def write_acs_bibliography_txt(df_doi, out_path="acs_bibliography.txt"):
    df = df_doi.sort_values("ref_id").copy()
    lines = [format_acs_ref(r) for _, r in df.iterrows()]
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(lines) + "\n")
    return out_path

# Generate outputs
docx_path = write_acs_bibliography_docx(df_join, "acs_complete_bibliography.docx")
txt_path  = write_acs_bibliography_txt(df_join, "acs_complete_bibliography.txt")

docx_path, txt_path

('acs_complete_bibliography.docx', 'acs_complete_bibliography.txt')

In [15]:
thresholds = PigThresholds(happy=50.0, maybe=35.0)

# df_compare (or df_join) must include: ref_id, match_score
# Optional: title, doi_url_final/doi_url, raw
build_acs_report(
    df_compare=df_join,  # or df_join
    out_pdf="Kilbreths_Pig_ACS_Report.pdf",
    article_title=ARTICLE_TITLE,
    source_pdf="test.pdf",
    thresholds=thresholds,
    logo_path="logo.png",
    yes_icon="yes.png",
    maybe_icon="maybe.png",
    no_icon="no.png",
)


'Kilbreths_Pig_ACS_Report.pdf'

In [16]:
# df_join = df_original.merge(
#     df_DOI[
#         [
#             "ref_id",
#             "title",
#             "authors",
#             "journal",
#             "journal_full",
#             "journal_iso4",
#             "journal_preferred",
#             "year",
#             "volume",
#             "issue",
#             "pages",
#             "doi",
#         ]
#     ],
#     on="ref_id",
#     how="left",
#     suffixes=("_orig", "_doi"),
# )


In [17]:
# tmp = df_join.apply(
#     lambda row: best_journal_score(
#         obs_journal=row["journal_orig"],
#         doi_journal_full=row["journal_full"],
#         doi_journal_iso4=row["journal_iso4"],
#         score_fn=fuzz.token_set_ratio,
#     ),
#     axis=1
# )

# df_join["journal_match_score"] = [t[0] for t in tmp]
# df_join["journal_match_variant"] = [t[1] for t in tmp]


In [18]:
# df_original["journal_obs"] = df_original["raw"].apply(extract_journal_from_acs_raw)


In [19]:
# df_join = df_original.merge(
#     df_DOI[
#         [
#             "ref_id",
#             "journal",
#             "journal_full",
#             "journal_iso4",
#             "journal_preferred",
#             "doi",
#             "title",
#             "authors",
#             "year",
#             "volume",
#             "issue",
#             "pages",
#         ]
#     ],
#     on="ref_id",
#     how="left",
#     suffixes=("_orig", "_doi")
# )


In [20]:
# # 1) Repair journal_obs using raw (row-wise)
# df_join["journal_obs"] = df_join.apply(
#     lambda r: clean_journal_obs(r.get("journal_obs",""), r.get("raw","")),
#     axis=1
# )

# # 2) Recompute journal matching scores
# tmp = df_join.apply(
#     lambda r: best_journal_score(
#         obs_journal=r.get("journal_obs",""),
#         doi_journal_full=r.get("journal_full",""),
#         doi_journal_iso4=r.get("journal_iso4",""),
#         score_fn=fuzz.token_set_ratio,
#     ),
#     axis=1
# )

# df_join["journal_match_score"]   = [t[0] for t in tmp]
# df_join["journal_match_variant"] = [t[1] for t in tmp]

# # 3) Inspect the problem row
# df_join.loc[df_join["ref_id"] == 23,
#             ["ref_id","journal_obs","journal_full","journal_iso4",
#              "journal_preferred","journal_match_score","journal_match_variant"]]




In [21]:
# obs = clean_journal_obs(row.get("journal_obs",""), row.get("raw",""))


In [22]:
# df_join["journal_obs"] = df_join.apply(
#     lambda r: clean_journal_obs(
#         r.get("journal_obs",""),
#         r.get("raw",""),
#         r.get("journal_full",""),
#         r.get("journal_iso4",""),
#     ),
#     axis=1,
# )

# df_join["journal_match_score"]   = [t[0] for t in tmp]
# df_join["journal_match_variant"] = [t[1] for t in tmp]

# # 3) Inspect the problem row
# df_join.loc[df_join["ref_id"] == 23,
#             ["ref_id","journal_obs","journal_full","journal_iso4",
#              "journal_preferred","journal_match_score","journal_match_variant"]]